# imports

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.special import gammaln
import pickle


1 unit eneregy = 1 K * k_b = 1.380649 × 10^{-23} J 

1 length unit = 1 nm = 10^{-9} m

1 mass unit = 10 elec mass = 9.1093837 × 7.5^{-30} kg

--> 1 time unit =  8.122745690860613e-13 s 

# The implementation of general methods for systems:

 - Initialization of common values
 - Setting initial positions randomly distributed inside the circle at 0, with a deviation of 1
 - Definition of the system's potential and the corresponding force
 - Plotting the potential surface as a heatmap, and plotting the force tensor field and plotting of the simulated trajectiries
 - Running the simulation for a given number of time steps

In [ ]:

class System:
    """
    A base class for simulating particle dynamics on a 2D double-well potential energy surface.

    This class sets up and manages an N-particle simulation in a configurable-dimensional
    space. It defines the potential energy landscape and corresponding force field, provides
    utilities for initializing particle positions, running dynamics (via subclasses), and
    visualizing results.

    The potential energy function is:
        U(x, y) = alpha * ((x² + y²)² + (x² - 9)²)
    which produces a double-well landscape with minima near (±3.5, 0).

    Parameters
    ----------
    num_steps : int, optional
        Number of simulation steps to run. Default is 100,000.
    num_particles : int, optional
        Number of particles in the simulation. Default is 1.
    dimension : int, optional
        Dimensionality of the simulation space. Default is 2.
        Note: potential energy and force methods assume 2D input.
    temperature : float, optional
        System temperature in Kelvin. Default is 300 K.
    alpha : float, optional
        Scaling factor for the potential energy surface. Default is 0.01.
    **kwargs
        Additional keyword arguments passed to parent class via super().__init__().

    Attributes
    ----------
    positions : np.ndarray, shape (num_particles, dimension)
        Current positions of all particles.
    trajectories : np.ndarray, shape (num_steps, num_particles, dimension)
        Recorded positions of all particles at each simulation step.

    Notes
    -----
    - This class is intended to be subclassed. The `__run__` method requires a
      `one_step()` method defined in a subclass to advance the simulation.
    - Call `set_initial_positions()` before `__run__()` to set starting positions.
    """
    def __init__(self, num_steps=int(1e5), num_particles=1, dimension=2, temperature=300, alpha=10**(-2), **kwargs):
        super().__init__(**kwargs)
        # Initialize simulation parameters
        self.num_particles = num_particles
        self.dimension = dimension
        self.num_steps = num_steps
        self.temperature = temperature  
        self.alpha = alpha
        # Initialize particle parameters
        self.positions = np.ones((num_particles, dimension))  
        self.trajectories = np.zeros((num_steps, num_particles, dimension))  #trajectory of each particle at each step

    def set_initial_positions(self, initial_positions=None):
            """externally set initial positions for particles, if None, random positions around the centre of grid will be generated
                must be used before updating the system"""
            if initial_positions is None:
                self.positions = np.random.normal(0, 1, size=(self.num_particles, self.dimension)) #normal distribution around center of grid with sd 1 
            else:
                self.positions = initial_positions
    
    def what_is_potential_energy(self, x):
        '''returns a potential energy at given position x for a one particle in 2D space
        Accepts either a 1D array-like of length 2 (single point) or an array with last dimension 2
        (e.g. grid with shape (..., 2)).'''
        pot = self.alpha * ( x[...,0]**4 + x[...,1]**4 - 2 * x[...,0]**2 - 4 * x[...,1]**2 + x[...,0] * x[...,1] + 0.3 * x[...,0] + 0.1 * x[...,1] )
        return pot

    def what_is_force(self, x):
        '''returns a force at given position x for a one particle in 2D space
        Accepts either a 1D array-like of length 2 (single point) or an array with last dimension 2
        (e.g. grid with shape (..., 2)).'''

        force_x = self.alpha *  ( -4 * x[...,0]**3 + 4 * x[...,0] - x[...,1] - 0.3 )
        force_y = -4 * self.alpha * ( x[...,1]**3 - 2 * x[...,1] + 0.25 * x[...,0] + 0.025 )
        return np.stack((force_x, force_y), axis=-1)

    def point_in_stable_region(self, point):
        """It returns True if the point is within one of the potential basins, i.e. if it is in the bottom right or top left pit. Otherwise, it returns False."""

        return np.where(self.what_is_potential_energy(point) < -4.2 * self.alpha, True, False)

    
    def plot_force_field(self, xlim=(-3, 3), ylim=(-3, 3), grid_points=20):
        '''plots the force field as a vector field'''
        x = np.linspace(xlim[0], xlim[1], grid_points)
        y = np.linspace(ylim[0], ylim[1], grid_points)
        X, Y = np.meshgrid(x, y)
        
        # Create position array with shape (grid_points, grid_points, 2)
        positions = np.stack((X, Y), axis=-1)
        
        # Calculate forces
        forces = self.what_is_force(positions)
        
        # Plot
        plt.figure(figsize=(10, 8))
        plt.quiver(X, Y, forces[..., 0], forces[..., 1])
        plt.title('Force Field')
        plt.xlabel('x')
        plt.ylabel('y')
        plt.grid()
        plt.show()

    def plot_potential_energy_surface(self, xlim=(-4, 4), ylim=(-4, 4)):
        '''plots the potential energy surface as a contour plot'''
        x=np.arange(xlim[0], xlim[1], 0.01)
        y=np.arange(ylim[0], ylim[1], 0.01)
        X, Y = np.meshgrid(x, y)
        
        # Create position array with shape (grid_points, grid_points, 2)
        positions = np.stack((X, Y), axis=-1)
        
        # Calculate potential energy
        Z = self.what_is_potential_energy(positions)
        
        # Plot
        plt.figure(figsize=(10, 8))
        plt.contourf(X, Y, Z, levels=100, cmap='hot')
        plt.colorbar(label='Potential Energy')
        plt.title('Potential Energy Surface')
        plt.xlabel('x')
        plt.ylabel('y')
        plt.show()
    

    def __run__(self):
        """
        Run the simulation and record particle trajectories.

        Must be called from a subclass that implements `one_step()`.

        First performs 5,000 equilibration steps (not recorded) to allow the
        system to relax from its initial configuration. Then runs `num_steps`
        production steps, recording positions at each step.

        If the subclass defines a momentum attribute `self.p`, momenta are also
        recorded and stored in `self.p_trajectories`.

        Raises
        ------
        AttributeError
            If `one_step()` is not defined in the subclass.

        Side Effects
        ------------
        Populates `self.trajectories` with shape (num_steps, num_particles, dimension).
        Optionally populates `self.p_trajectories` with the same shape.
        """
        for i in range(5000):
             self.one_step()
            

        try:
            self.p
            trajectory_p = np.zeros((self.num_steps, self.num_particles, self.dimension))
            for i in tqdm(range(self.num_steps)):
                self.trajectories[i] = self.positions
                trajectory_p[i] = self.p
                self.one_step()
            self.p_trajectories = trajectory_p
        except:
            for i in tqdm(range(self.num_steps)):
                self.one_step()
                self.trajectories[i] = self.positions

    def save_trajectories(self, filename='trajectories'):
        '''saves the trajectories to a numpy file'''
        np.save(filename+'.npy', self.trajectories)

    def load_trajectories(self, filename='trajectories'):
        '''loads the trajectories from a numpy file'''
        self.trajectories = np.load(filename+'.npy')

    def plot_point_density(self):
        '''plots the trajectories of the 2 dimensional particles as a heatmap of point density'''
        x = self.trajectories[:, :, 0].flatten()
        y = self.trajectories[:, :, 1].flatten()
        # 3. Create the Heatmap
        plt.figure(figsize=(8, 6))

        # bins: controls the resolution of the heatmap
        # cmap: 'hot', 'inferno', 'magma', or 'viridis' are great for heatmaps
        plt.hist2d(x, y, bins=500, cmap='gist_earth')

        # Add aesthetics
        plt.colorbar(label='Point Density')
        
        plt.title('2D Point Density for alpha='+str(self.alpha) + ' with ' + str(self.trajectories.shape[0]) + ' steps')
        plt.xlabel('X Coordinate')
        plt.ylabel('Y Coordinate')
        plt.show()

    


    




## Example of usage

In [ ]:
test = System(alpha = 1)
test.plot_potential_energy_surface( xlim= (-3, 3), ylim = (-3, 3) )

pot_value_grid = np.array([[test.what_is_potential_energy(np.array([x, y])) for x in np.arange(-2, 2, 0.1)] for y in np.arange(-2, 2, 0.1)])
plt.figure(figsize=(10, 8))
plt.contourf(np.arange(-2.5, 2.5, 0.1), np.arange(-2.5, 2.5, 0.1), pot_value_grid, levels=150, vmax = 25, cmap='hot')
plt.colorbar(label='Potential Energy')
print("ENERGY AT -1 0 is", test.what_is_potential_energy(np.array([-1, 0])))

X, Y = np.meshgrid(np.arange(-2.5, 2.5, 0.1), np.arange(-2.5, 2.5, 0.1))
points_index  = test.point_in_stable_region( np.stack((X, Y), axis=-1) )
plt.scatter(X[points_index], Y[points_index], color='white', label='Stable Region', s=5)
plt.legend()
plt.title('Potential Energy Surface')
plt.xlabel('x')
plt.ylabel('y')
plt.tight_layout()
plt.show()
test = System(alpha = 165.289)
pot_value_grid = np.array([[test.what_is_potential_energy(np.array([x, y])) for x in np.arange(-2, 2, 0.1)] for y in np.arange(-2, 2, 0.1)])

test.plot_force_field( xlim= (-2, 2), ylim = (-2, 2) )



# Implementation of the Monte Carlo method in the configuration space

including the method for the simulation of one time step.

In [ ]:
class MonteCarloSimulator(System):
    def __init__(self, step_size=0.5, **kwargs):
        super().__init__(**kwargs)
        self.beta = 1 / self.temperature
        self.step_size = step_size
        self.pot=np.ones(self.num_particles)*(np.inf)

    def one_step(self):
        for i in range(self.num_particles): #updating each particle one by one
            displacement = (np.random.ranf(self.dimension)-0.5) * self.step_size
            pot=self.what_is_potential_energy(self.positions[i]+displacement) #calculate potential energy at new position
            #ACCEPTANCE CRITERION
            if self.pot[i] > pot:
                self.positions[i] += displacement
                self.pot[i]=pot
            elif np.random.rand() < np.exp(-self.beta * (pot - self.pot[i])):
                self.positions[i] += displacement
                self.pot[i]=pot

    def run(self):
        self.__run__()
        
    
    

# Implementation of the Molecular Dynamics methods:

- Simulation of one time step according to the Langevin dynamics using the BAOAB algorithm
- Calculating the ratio between the mean squared error of total energy and the mean squared error of potential energy
- Plotting energies versus time and temperatures versus time

In [ ]:
class Molecular_dynamics(System):
    """Molecular Dynamics (MD) simulation environment extending a base physical system.

    This class manages particle trajectories, initial thermal momentum sampling, and 
    Langevin thermostat parameters for a molecular system. It inherits foundational 
    physical properties (such as particle positions, dimensions, and temperature) 
    from the parent `System` class.

    Parameters
    ----------
    dt : float, optional
        Integration time step $\Delta t$ in simulation time units. Default is 0.001.
    m : float, optional
        Mass $m$ of each particle. Default is 0.1.
    gamma : float, optional
        Friction/damping coefficient $\gamma$ for Langevin dynamics. A value of 0 
        corresponds to pure Hamiltonian (NVE) dynamics. Default is 0.
    **kwargs
        Arbitrary keyword arguments passed to the parent `System` class initializer 
        (e.g., `num_particles`, `dimension`, `temperature`, `positions`).

    Attributes
    ----------
    dt : float
        Integration time step.
    gamma : float
        Friction/damping coefficient.
    m : float
        Particle mass.
    p : np.ndarray
        Array of shape `(num_particles, dimension)` containing initial particle momenta, 
        sampled from a Maxwell-Boltzmann thermal distribution scaled by particle mass 
        and temperature.
    zeta : float
        Thermal noise scaling factor for stochastic integration steps, computed as:
        $$\zeta = \sqrt{m \cdot T \cdot (1 - e^{-2 \gamma \Delta t})}$$
    current_force : np.ndarray
        Array of shape matching `positions` containing the initial forces acting on 
        the particles at time $t = 0$.

    Notes
    -----
    * **Parent Inheritance:** The parent `System` class initializer must set 
      `self.temperature`, `self.num_particles`, `self.dimension`, and `self.positions` 
      prior to the execution of `Molecular_dynamics.__init__`.
    * **Force Evaluation:** Relies on the instance method `self.what_is_force()` 
      (inherited from `System` or defined on the subclass) to evaluate initial forces.
    """
    def __init__(self, dt=1e-3, m=0.1, gamma=0, **kwargs):
        super().__init__(**kwargs)
        self.dt = dt
        self.gamma = gamma
        self.m = m
        self.p = m * np.random.normal(0, (m*self.temperature)**(0.5), size=(self.num_particles, self.dimension))
        self.zeta = np.sqrt( self.m * self.temperature * (1 - np.exp(-2 * self.gamma * self.dt)) )
        self.current_force = self.what_is_force(self.positions)

    def one_step(self):
        """updates the system by one time step using the velocity BAOAB algorithm"""
        #for particle in range (self.num_particles):
        temp_q = self.positions #To avoid calling self.position() multiple times
        temp_p = self.p
        # --- B: half-kick ---
        temp_p = temp_p + 0.5 * self.dt * self.current_force #Particles are not interacting with each other, so we can calculate forces for one specific particle at a time 
        # --- A: half-drift ---
        temp_q = temp_q + 0.5 * self.dt * temp_p / self.m 
        # --- O: Ornstein-Uhlenbeck ---
        strange_stuff = np.exp(-self.gamma * self.dt) * temp_p + self.zeta * np.random.normal(0, 1, size=(self.num_particles, self.dimension))
        # --- A: half-drift ---
        self.positions = temp_q + 0.5 * self.dt * strange_stuff / self.m #so that self.positions is already the new displaced position 
        # --- B: half-kick ---
        self.current_force = self.what_is_force(self.positions) #I would like to make it an output variable, but to do so, I must change the __run__ method, which means it won't be as flexible as I want.
        self.p = strange_stuff + 0.5 * self.dt * self.current_force 

    def run(self):
        self.__run__()

    def pot_error_to_full_energy(self, trajectory_q=None, trajectory_p=None, simulate=True):
        """calculates the error of potential energy to the full energy for each time step and returns the mean"""
        if simulate:
            self.__run__()
            trajectory_q, trajectory_p = self.trajectories[:, 0, :], self.p_trajectories[:, 0, :]

        potential_energy = self.what_is_potential_energy(trajectory_q)
        kinetic_energy = 0.5  * np.sum(trajectory_p**2, axis=-1) / self.m
        total_energy = potential_energy + kinetic_energy

        potential_energy = potential_energy.flatten()
        total_energy = total_energy.flatten()

        potential_energy_error_sq = (np.mean(potential_energy) - potential_energy)**2
        total_energy_error_sq = (total_energy - np.mean(total_energy))**2
        energy_error = np.mean(total_energy_error_sq / potential_energy_error_sq )
        return energy_error
    
    def plot_energies_vs_time(self, trajectory_q=None, trajectory_p=None, simulate=True, separate = False):
        """
        Plot potential, kinetic, and total energies as a function of time steps.

        If `simulate=True`, runs the simulation internally and uses the resulting
        trajectories. Otherwise, expects pre-computed position and momentum
        trajectories to be passed directly.

        Parameters
        ----------
        trajectory_q : np.ndarray, optional
            Position trajectory array of shape (steps, dims). Required if simulate=False.
        trajectory_p : np.ndarray, optional
            Momentum trajectory array of shape (steps, dims). Required if simulate=False.
        simulate : bool, optional
            If True (default), runs the simulation via __run__() before plotting.
        separate : bool, optional
            If True, renders two side-by-side subplots — one for total energy,
            one for potential and kinetic energies. If False (default), all three
            curves are overlaid on a single plot.
        """
        
        if simulate:
            initial_pot_energy = self.what_is_potential_energy(self.positions[0])#The potential energy at the initial position of the first particle
            self.__run__()
            trajectory_q, trajectory_p = self.trajectories[:, 0, :], self.p_trajectories[:, 0, :]
        else:
            initial_pot_energy = self.what_is_potential_energy(trajectory_q[0])#The potential energy at the initial position of the first particle
        
        potential_energy = self.what_is_potential_energy(trajectory_q)
        kinetic_energy = 0.5 * np.sum(trajectory_p**2, axis=-1) / self.m
        total_energy = kinetic_energy + potential_energy 

        potential_energy = potential_energy.flatten()
        kinetic_energy = kinetic_energy.flatten()
        total_energy = total_energy.flatten()

        steps = np.arange(potential_energy.size)

        if separate:
            fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(16, 6))

            
            ax_left.plot(steps, total_energy, label='Total energy')
            ax_left.plot(steps, total_energy-initial_pot_energy, label='Total energy - initial potential energy', alpha=0.3)
            ax_left.set_xlabel('Time step')
            ax_left.set_ylabel('Energy')
            ax_left.legend()
            ax_left.set_title('Total energy vs time for dt = '+str(self.dt)+' time units and for gamma = '+str(self.gamma))
            ax_left.grid(alpha=0.3) 

            ax_right.plot(steps, potential_energy, label='Potential energy', alpha=0.75)
            ax_right.plot(steps, kinetic_energy, label='Kinetic energy', alpha=0.75)
            ax_right.set_xlabel('Time step')
            ax_right.set_ylabel('Energy')
            ax_right.set_title('Energy vs time')
            ax_right.legend()
            ax_right.grid(alpha=0.3) 
            fig.tight_layout()
            plt.show()

        else:
            plt.figure(figsize=(8, 6))
            plt.plot(steps, potential_energy, label='Potential energy', alpha=0.5)
            plt.plot(steps, kinetic_energy, label='Kinetic energy', alpha=0.5)
            plt.plot(steps, total_energy, label='Total energy', alpha=0.75)
            plt.plot(steps, np.ones_like(steps) * initial_pot_energy, label='Initial potential energy', alpha=0.3)
            plt.xlabel('Time step')
            plt.ylabel('Energy')
            plt.title('Energy vs time for dt = '+str(self.dt)+' time units and for gamma = '+str(self.gamma))
            plt.legend()
            plt.grid(alpha=0.3) 
            plt.show()
        
    
    def what_is_temperature(self, energy=None):
        """
        Estimate the system temperature from the current microcanonical energy state

        Returns
        -------
        float
            Estimated temperature in reduced units.
        """
        if len(energy.shape)==1:
            return self.dimension * energy / 2
        elif len(energy.shape)==2:
            return self.dimension * np.mean(energy, axis=-1) / 2

        """
        For free particle
        if (p==None).any():
            return self.dimension * np.mean(np.linalg.norm(self.p, axis=-1)**2) / (self.m * 4) #the kinetic energy = <p**2> / 2m = N_dim * k_b * T / 2
        else:
            sub_result = np.linalg.norm(p, axis=-1)**2
            return self.dimension * np.mean(sub_result , axis=-1) / (self.m * 4)
        """
            
    def plot_temperature_vs_time(self, trajectory_p=None, trajectory_q = None, simulate=True):
        """Calculate and plot the instantaneous temperature evolution over time steps.

        This method computes the time-dependent temperature profile of a simulation system. 
        It can either execute a new simulation run via `self.__run__()` or compute energies 
        from pre-calculated position and momentum trajectory arrays supplied as arguments.

        The temperature is derived by:
        1. Calculating kinetic energy $E_{\text{kin}} = \frac{p^2}{2m}$ from particle momenta.
        2. Computing total effective energy relative to the initial potential energy baseline.
        3. Mapping the adjusted energy to temperature via `self.what_is_temperature()`.

        Parameters
        ----------
        trajectory_p : np.ndarray or None, optional
            A NumPy array of particle momenta over time. Required if `simulate=False`. 
            Ignored if `simulate=True`. Default is `None`.
        trajectory_q : np.ndarray or None, optional
            A NumPy array of particle positions over time. Required if `simulate=False`. 
            Ignored if `simulate=True`. Default is `None`.
        simulate : bool, optional
            If `True`, triggers `self.__run__()` to execute a new simulation and populates 
            trajectories internally from `self.trajectories` and `self.p_trajectories`. 
            If `False`, uses the provided `trajectory_q` and `trajectory_p` arrays. 
            Default is `True`.

        Returns
        -------
        None
            The method does not return any values; it directly renders and displays 
            the matplotlib figure.

        Notes
        -----
        * **Class Dependencies:** This method relies on several instance attributes and methods:
        - Attributes: `self.m` (particle mass), `self.gamma` (friction coefficient/damping parameter),
            `self.positions`, `self.trajectories`, `self.p_trajectories`.
        - Methods: `self.what_is_potential_energy()`, `self.what_is_temperature()`, and `self.__run__()`.
        * **Energy Baseline:** The total energy calculation subtracts `initial_pot_energy` 
        (the potential energy at step 0 for the first particle) as a reference offset before 
        evaluating the temperature.
        * **Plot Styling:** Temperature points are plotted against integer time steps as fine, 
        semi-transparent scatter markers (`marker='x'`, `s=0.1`) to clearly display high-density 
        time series data without clutter.
        """
        if simulate:
            initial_pot_energy = self.what_is_potential_energy(self.positions[0])#The potential energy at the initial position of the first particle
            self.__run__()
            trajectory_q, trajectory_p = self.trajectories[:, 0, :], self.p_trajectories[:, 0, :]
        else:
            initial_pot_energy = self.what_is_potential_energy(trajectory_q[0, 0])#The potential energy at the initial position of the first particle
        

        potential_energy = self.what_is_potential_energy(trajectory_q)
        kinetic_energy = 0.5 * np.sum(trajectory_p**2, axis=-1) / self.m
        total_energy =  kinetic_energy + potential_energy - initial_pot_energy

        temperature = self.what_is_temperature(total_energy)
        time =  np.arange(temperature.size)

        plt.figure(figsize=(8, 6))
        plt.scatter(time, temperature, label='Temperature calculated from particle momenta', marker= 'x', s=0.1)
        plt.xlabel('Time step')
        plt.ylabel('Temperature')
        plt.title('Temperature vs time for gamma = '+str(self.gamma))
        plt.legend()
        plt.grid(alpha=0.3) 
        plt.show()


    

        


# Definition of the TPS class and its associated methods:
- Extraction of a transition path from the first particle's trajectory
- Creation of a transition path (not universal; needs to be modified)
- Running the equilibrium simulation and extracting all transition paths from the simulated trajectory
- Plotting of the transition path, the committor and the configuration density, showing which configurations do not belong to the TPE
- Creation of a committor and a new candidate path using the shooting algorithm

In [ ]:
class Transition_Path_Sampler(Molecular_dynamics, MonteCarloSimulator):
    """Transition Path Sampling (TPS) ensemble manager for reactive trajectory generation and analysis.

    This class inherits from both `Molecular_dynamics` and `MonteCarloSimulator` to provide 
    a flexible ensemble generator for rare-event sampling. It supports trajectory extraction, 
    two-way shooting moves, committor surface calculation, and visualization of transition 
    paths across underlying potential energy landscapes.

    Parameters
    ----------
    molec_dynam_step : bool, optional
        If True, selects `Molecular_dynamics.one_step` as the underlying propagation engine.
        If False, selects `MonteCarloSimulator.one_step`. Default is True.
    **all_args
        Arbitrary keyword arguments passed directly to the parent class initializers 
        (`Molecular_dynamics` and `MonteCarloSimulator`).

    Attributes
    ----------
    _step_fn : callable
        Bound method pointing to either MD or MC single-step integration/trial routines.

    Notes
    -----
    * **Multiple Inheritance:** Initializes both MD and MC base functionality. If any passed 
      argument fails validation during initialization, a `TypeError` is caught and logged.
    """
    def __init__(self, molec_dynam_step = True, **all_args):
        try:
            super().__init__(**all_args)
            if molec_dynam_step:
                self._step_fn = Molecular_dynamics.one_step
            else:
                self._step_fn = MonteCarloSimulator.one_step
        except TypeError:
            print("Some of the arguments were incorrect. The Transition_Path_Sampler object was not initialised. Try again.")

    def one_step(self):
        self._step_fn(self)

    def extract_original_transition_path(self, raw_transition_path=np.zeros(2), return_last_index=False):
        """It looks for transition paths inside raw_transition_path. 
        If the raw_transition_path is a three-dimensional array, it crashes. 
        If no raw_transition_path is given, the function uses the simulated trajectory of the FIRST particle.  """
        if (raw_transition_path == 0).all():
            try:
                raw_transition_path = (self.trajectories[:, 0, :])
            except:
                raw_transition_path = self.trajectories
        stable_mask = self.point_in_stable_region(raw_transition_path)
        
        # Search for the points in the stable region
        entry_idx = np.where(stable_mask)[0]
        # If there are no entries into the stable region, return an empty array
        if len(entry_idx) == 0:
            return np.array([])
        
        start_idx = entry_idx[0] #First point in the stable region
        first_pos = raw_transition_path[start_idx]
        right = first_pos[0] > 0 #Determine if the particle is in the right or left well based on the x-coordinate of the first point in the stable region
        
        # Find transition to opposite well
        for j in range(start_idx + 1, len(raw_transition_path)):
            if stable_mask[j]: # If the particle is in the stable region, check if it has transitioned to the opposite well
                curr_pos = raw_transition_path[j]
                if (right and curr_pos[0] < 0) or (not right and curr_pos[0] > 0):
                    if return_last_index:
                        return raw_transition_path[start_idx:j+1], j+1
                    else:
                        return raw_transition_path[start_idx:j+1]
                elif (right and curr_pos[0] > 0) or (not right and curr_pos[0] < 0): 
                    start_idx = j  # Update start index if particle goes back to the original well
        
        if return_last_index:
            return None, None
        print("No transition path was found")

    def create_transition_path(self, Points_num, step_size = 0.25):
        """It must be tuned to arbitrary potentials and stable states."""
        x = np.arange(-1.65-step_size, 1.65+step_size, step_size)#last point is not included
        raw_path_length = len(x)
        raw_trasition =  np.array( (x, np.zeros(raw_path_length))).T #line between (-1.65, 0) and (1.65, 0)
        points_left=Points_num - raw_path_length
        if points_left>0:
            for i in range(points_left):
                split_point=np.random.randint(low=2, high=raw_path_length-2)
                #Calculation of rotation matrix elements
                cos_alpha = (raw_trasition[split_point+1][0] - raw_trasition[split_point-1][0] ) / np.linalg.norm(raw_trasition[split_point+1] - raw_trasition[split_point-1])#Calculate the cosine between the vector (split_point + 1, split_point – 1) and the vector (-1, 0).
                sin_alpha = (raw_trasition[split_point+1][1] - raw_trasition[split_point-1][1] )/ np.linalg.norm(raw_trasition[split_point+1] - raw_trasition[split_point-1])
                rotation_matrix = np.array( ((cos_alpha, sin_alpha), (-sin_alpha, cos_alpha)))#clockwise rotation matrix
                if np.random.choice([True, False]):#Choose randomly whether it splits in the +y or -y direction.
                    old_point=raw_trasition[split_point].copy()
                    raw_trasition[split_point] = old_point - np.array((step_size/2 , step_size * np.sqrt(3)/2))
                    raw_trasition[split_point] = np.dot(rotation_matrix, raw_trasition[split_point])
                    raw_trasition = np.insert(raw_trasition, split_point+1, old_point + np.array((step_size/2 , -step_size * np.sqrt(3)/2)), axis=0 )
                    raw_trasition[split_point+1] = np.dot(rotation_matrix, raw_trasition[split_point+1])
                else:
                    old_point=raw_trasition[split_point].copy()
                    raw_trasition[split_point] = old_point - np.array((step_size/2 , -step_size * np.sqrt(3)/2))
                    raw_trasition[split_point] = np.dot(rotation_matrix, raw_trasition[split_point])
                    raw_trasition = np.insert(raw_trasition, split_point+1, old_point + np.array((step_size/2 , step_size * np.sqrt(3)/2)), axis=0)
                    raw_trasition[split_point+1] = np.dot(rotation_matrix, raw_trasition[split_point+1])
            return raw_trasition
        elif points_left<0:
            print("This transition path is impossible.")
        else:
            return raw_trasition
        
    def simulate_extract_all_transition_paths(self, save=True, file_name=False, times=1, return_num_of_transitions = False):
        """
        Run the simulation and extract all transition paths, discarding the rest of the trajectory.

        Runs the simulation `times` times, each time extracting every reactive segment
        (i.e. contiguous trajectory segments that connect the two stable basins) and
        either saving them to disk or returning them directly.

        Parameters
        ----------
        save : bool, optional
            If True (default), save each run's transition paths to an .npy file.
            If False, return the concatenated paths from the first run instead.
        file_name : str or False, optional
            Base name for the output file. If False (default), a name is generated
            automatically from self.num_steps, self.alpha, and self.temperature.
            Each run's file is suffixed with a zero-padded run index, e.g. ' run no 03.npy'.
        times : int, optional
            Number of independent simulation runs to perform. Default is 1.
        return_num_of_transitions : bool, optional
            Only used when save=False. If True, also return the number of transition
            paths found alongside the coordinate array. Default is False.

        Returns
        -------
        transition_paths_coordinates : np.ndarray, shape (n, 2)
            Concatenated coordinates of all transition path points across the run.
            Only returned when save=False.
        transitions_counter : int
            Number of distinct transition paths found in the run.
            Only returned when save=False and return_num_of_transitions=True.

        Side Effects
        ------------
        When save=True, writes one .npy file per run containing an array of shape
        (n, 2), where n is the total number of transition path points across that run.

        Notes
        -----
        When save=False, only the first run's results are returned — the `times`
        parameter has no practical effect in this case.
        """

        num_of_digits = len(str(times))
        
        for i in range(times):
            self.set_initial_positions()
            self.__run__()
            current_start_index=0
            transitions_counter = 0
            transition_paths=[]
            last_index = self.trajectories.shape[0]
            while current_start_index < last_index:
                transition_path, temp_start_index = self.extract_original_transition_path(self.trajectories[current_start_index: , 0, :], return_last_index=True)
                if temp_start_index != None: #I could not think of a more efficient way than this little condition
                    current_start_index += temp_start_index
                    transition_paths.append(transition_path)
                    transitions_counter += 1 
                else:
                    current_start_index = last_index
            transition_paths_coordinates = np.concatenate(transition_paths)
            
            if save:
                if not file_name: #If file_name were a string value, it would be true without not
                    file_name='The transition paths for '+str(self.num_steps)+' number of steps and for alpha = '+str(self.alpha)+' by T = '+str(self.temperature)+ f' run no {i:0{num_of_digits}d}.npy'
                else:
                    file_name = file_name + f' run no {i:0{num_of_digits}d}.npy'
                file_name = file_name[:-(4+num_of_digits)] 
                file_name += f'{i:0{num_of_digits}d}' + '.npy'
                np.save(file_name, transition_paths_coordinates)
                if return_num_of_transitions:
                    return transitions_counter
            else:
                if return_num_of_transitions:
                    return transition_paths_coordinates, transitions_counter
                else:
                    return transition_paths_coordinates
                
            

    def plot_transition_path(self, transition_path,  plot_potential=True, plot_stable_region=True):
        '''plots the transition path'''
        try:
            if transition_path == None:
                print("FAIL")
                return 0 
        except:
            num_steps = transition_path.shape[0]
            print("number of steps for transition:", num_steps)
        x = transition_path[:, 0]
        y = transition_path[:, 1]
        
        
        plt.figure(figsize=(8, 6))
        plt.scatter(x, y, c=range(num_steps), cmap='cool', s=1, alpha=0.5)
        plt.colorbar(label='Step')
        plt.scatter(x[0], y[0], c='red', s=10, label='Start')
        plt.scatter(x[-1], y[-1], c='blue', s=10, label='End')
        if plot_potential:
            X, Y = np.meshgrid(np.arange(-2, 2, 0.1), np.arange(-2, 2, 0.1), indexing='xy')
            points = np.stack((X, Y), axis=-1)  # shape (len(y), len(x), 2)
            Z = self.what_is_potential_energy(points)
            plt.contour(X, Y, Z, levels=40, colors='black', linewidths=0.5, alpha=0.3)
            
        if plot_stable_region:
            X, Y = np.meshgrid(np.arange(-2, 2, 0.1), np.arange(-2, 2, 0.1), indexing='xy')
            points = np.stack((X, Y), axis=-1)  # shape (len(y), len(x), 2)
            stable_mask = self.point_in_stable_region(points)
            plt.scatter(X[stable_mask], Y[stable_mask], color='green', label='Stable Region (U < -5)', s=5)
        # Add aesthetics
        plt.title('Transition Path for alpha=' + str(self.alpha))
        plt.xlabel('X Coordinate')
        plt.ylabel('Y Coordinate')
        plt.grid(alpha=0.1)
        plt.legend()
        plt.show()
    
    def plot_point_density_minus_transition(self, plot_potential=True, simulate=True, show_transition_path_over=True, precise=False):
        """Calculate and visualize spatial point density of trajectory frames excluding transition paths.

        This method identifies all reactive transition paths within a trajectory (either 
        generated on-the-fly or pre-existing in `self.trajectories`), calculates 2D spatial 
        histograms for both the full trajectory and the combined transition path frames, and 
        subtracts the transition path counts from the total counts ($hist_1 - hist_2$). 

        The resulting density map highlights non-reactive spatial exploration and dwelling 
        time within metastable basins, isolating unreactive motion from reactive crossings.

        Parameters
        ----------
        plot_potential : bool, optional
            If True, evaluates `self.what_is_potential_energy()` over a $[-4, 4] \times [-4, 4]$ 
            spatial grid and overlays equipotential contour lines. Default is True.
        simulate : bool, optional
            If True, re-initializes positions via `self.set_initial_positions()` and runs 
            a new simulation sweep via `self.__run__()` prior to processing. If False, 
            evaluates existing data in `self.trajectories`. Default is True.
        show_transition_path_over : bool, optional
            If True, overlays extracted transition path coordinates as semi-transparent red 
            scatter points on top of the non-reactive density heatmap. Default is True.
        precise : bool, optional
            Controls the spatial resolution of the 2D histogram. If True, uses 500 bins 
            per dimension; if False, uses 100 bins. Default is False.

        Returns
        -------
        None
            The method does not return any values; it directly renders and displays 
            the matplotlib figure.

        Notes
        -----
        * **Class Dependencies:** Relies on several instance attributes and methods:
        - Attributes: `self.trajectories` (array of shape `(n_steps, n_particles, dimension)`), 
            `self.alpha`, and `self.num_steps`.
        - Methods: `self.set_initial_positions()`, `self.__run__()`, 
            `self.extract_original_transition_path()`, and `self.what_is_potential_energy()`.
        * **Histogram Alignment:** `hist2` uses the exact spatial bounding edges (`xedges`, `yedges`) 
        computed from `hist1` to guarantee pixel-wise bin alignment during matrix subtraction.
        * **Zero Handling:** Grid bins with zero density differences ($hist_1 - hist_2 = 0$) 
        are masked as `np.nan` to render unvisited spatial regions transparently in 
        `plt.pcolormesh`.
        """
        if simulate:
            self.set_initial_positions()
            self.__run__()
        current_start_index=0
        transitions_counter = 0
        transition_paths=[]
        last_index = self.trajectories.shape[0]
        while current_start_index < last_index:
            transition_path, temp_start_index = self.extract_original_transition_path(self.trajectories[current_start_index: , 0, :], return_last_index=True)
            if temp_start_index != None: #I could not think of a more efficient way than this little condition
                current_start_index += temp_start_index
                transition_paths.append(transition_path)
                transitions_counter += 1 
            else:
                current_start_index = last_index

        #np.concatenate(transition_paths) converts the Python list to an Numpy array, as if transition_paths.flatten() were used.
        transition_paths_coordinates = np.concatenate(transition_paths)
        plt.figure(figsize=(8, 6))
        x1 = self.trajectories[:, 0, 0].flatten()
        y1 = self.trajectories[:, 0, 1].flatten()
        x2 = transition_paths_coordinates[:, 0].flatten() 
        y2 = transition_paths_coordinates[:, 1].flatten()
        if precise:
            bins=500
        else:
            bins=100

        hist1, xedges, yedges = np.histogram2d(x1, y1, bins=bins)
        hist2, _, _ = np.histogram2d(x2, y2, bins=bins, range=[[xedges[0], xedges[-1]], [yedges[0], yedges[-1]]])
        diff = hist1 - hist2
        diff = np.where(diff == 0, np.nan, diff)  # Set zero differences to NaN for better visualization
        #Check if it would also show the transition path so if subtraction works. If shows, "diff" would replace hist2, and we wouldn't see any red dots.
        #hist2iff = np.where(hist2 == 0, np.nan, hist2)
        #plt.pcolormesh(xedges, yedges, hist2iff.T, cmap='autumn')
        im = plt.pcolormesh(xedges, yedges, diff.T, cmap='summer')
        if show_transition_path_over:
            plt.scatter(x2, y2, c='red', s=0.5, alpha=0.75)

        #plot equipotential lines
        if plot_potential:
            X, Y = np.meshgrid(np.arange(-4, 4, 0.5), np.arange(-4, 4, 0.5), indexing='xy')
            points = np.stack((X, Y), axis=-1)  # shape (len(y), len(x), 2)
            Z = self.what_is_potential_energy(points)
            plt.contour(X, Y, Z, levels=40, colors='black', linewidths=0.5, alpha=0.25)

        # Add aesthetics
        plt.colorbar(im, label='Density')
        plt.title('Point Density - transition paths for alpha='+str(self.alpha) + ' with ' + str(self.num_steps) + ' steps'+ ' and ' + str(transitions_counter) + ' transition paths')
        plt.xlabel('X Coordinate')
        plt.ylabel('Y Coordinate')
        plt.show()

        
    def Create_a_transition_path_with_shooting(self, original_transition_path=np.zeros(2), shooting_point_idx = None, return_Ns =False) -> np.ndarray:
        """Generate a new candidate transition path using two-way shooting and Metropolis acceptance.

        This method implements a two-way shooting move in Transition Path Sampling (TPS). 
        It selects a configuration (shooting point) along an existing valid path, initializes 
        two walkers with opposite equal-magnitude Maxwell-Boltzmann momenta ($p$ and $-p$), 
        and integrates them step-by-step until both enter stable basins. 

        The move involves two distinct propagation phases:
        1. **Phase 1 (Dual Propagation):** Both forward and backward walkers propagate 
        simultaneously until the first walker reaches a stable basin.
        2. **Phase 2 (Single Propagation):** The remaining walker continues integration 
        until it hits a stable basin.

        If both walkers commit to the *same* basin, the candidate trajectory is non-reactive 
        and is immediately rejected. If the path connects opposite basins (Basin A and Basin B), 
        it is accepted according to the Metropolis-Hastings path length ratio:

        .. math:: P_{\\text{acc}} = \\min\\left(1, \\frac{L_{\\text{old}}}{L_{\\text{new}}}\\right)

        Parameters
        ----------
        original_transition_path : np.ndarray, optional
            A 2D float array of shape `(N, 2)` representing the reference reactive path. 
            If uninitialized (e.g., default `np.zeros(2)`), defaults to evaluating 
            `self.trajectories`.
        shooting_point_idx : int or None, optional
            Frame index along `original_transition_path` from which to launch the shooting move. 
            If `None`, an index is sampled uniformly at random from $[0, N-1]$. Default is `None`.
        return_Ns : bool, optional
            If `True`, returns the trajectory along with the commitment counts $N_A$ and $N_B$ 
            (the number of shooting walkers that terminated in Basin A and Basin B, respectively). 
            Default is `False`.

        Returns
        -------
        new_path : np.ndarray
            A 2D float array of shape `(M, 2)` containing coordinates of the accepted trajectory. 
            If the shooting move fails or is rejected by the Metropolis criterion, 
            returns `original_transition_path`.
        N_a : int, optional
            The number of shooting walkers that reached Basin A ($x \\le 0$). Only returned if 
            `return_Ns=True`.
        N_b : int, optional
            The number of shooting walkers that reached Basin B ($x > 0$), computed as $2 - N_a$. 
            Only returned if `return_Ns=True`.

        Notes
        -----
        * **Buffer Allocation:** Pre-allocates a trajectory buffer of length 
        `new_len = int(self.alpha * 10^6 / self.temperature)` to prevent dynamic array resizing 
        during step-by-step integration.
        * **Basin Boundaries:** Basin A is identified by $x \\le 0$ and Basin B by $x > 0$ 
        at the moment `self.point_in_stable_region()` evaluates to `True`.
        * **Rejection Conditions:**
        - **Non-reactive:** Both walkers land in the same basin (returns `original_transition_path`).
        - **Metropolis:** $L_{\\text{old}} / L_{\\text{new}} < U(0, 1)$ (returns `original_transition_path`).
        * **State Mutation:** Temporarily alters instance attributes `self.num_particles`, 
        `self.positions`, `self.p`, and `self.current_force` during propagation. 
        `self.num_particles` is restored upon completion.
        """
        if (original_transition_path == 0).all():
            try:
                original_transition_path = self.trajectories[:, 0, :]
            except:
                original_transition_path = self.trajectories

        if return_Ns:
            N_a = 0

        length_O  = original_transition_path.shape[0]
        if shooting_point_idx is None:
            shooting_point_idx = np.random.randint(length_O)

        current_pos = original_transition_path[shooting_point_idx]

        new_len    = int(self.alpha * (10 ** 6) / self.temperature)
        new_transit = np.empty((new_len, 2), dtype=np.float64)
        start_index   = new_len // 2
        new_transit[start_index] = current_pos

        save_particle_num  = self.num_particles
        self.num_particles = 2
        self.positions     = np.array([current_pos, current_pos])
        p_eq               = self.m * np.random.normal(0, (self.m * self.temperature) ** 0.5, size=2)
        self.p             = np.array([p_eq, -p_eq])    
        self.current_force = self.what_is_force(self.positions)

        # ── Phase 1: find which walker reaches a basin first ──────────────────
        if shooting_point_idx == 0 or shooting_point_idx == length_O - 1:
            # One position is already at a basin endpoint
            if self.point_in_stable_region(self.positions[1]):
                # 2 = backward reached left/A basin, 3 = backward reached right/B basin
                right = int(self.positions[1, 0] > 0) + 2
            else:
                # 0 = forward reached left/A basin, 1 = forward reached right/B basin
                right = int(self.positions[0, 0] > 0)
            finish1_index = start_index
        else:
            while start_index !=0:
                self.one_step()
                start_index -= 1
                q_front = self.positions[0]
                q_back  = self.positions[1]
                new_transit[start_index]           = q_front
                new_transit[new_len - start_index] = q_back
                if self.point_in_stable_region(q_back):
                    right         = int(q_back[0] > 0) + 2
                    finish1_index = new_len - start_index
                    break
                if self.point_in_stable_region(q_front):
                    right         = int(q_front[0] > 0)
                    finish1_index = start_index
                    break
        
        if return_Ns:
            if right % 2 ==0:
                N_a += 1

        # ── Phase 2: continue the other walker to the opposite basin ──────────
        # right 0/1 → front finished → drop index 0, store mirrored
        # right 2/3 → back  finished → drop index 1, store forward
        removing_index = right // 2          # 0 for right ∈ {0,1}, 1 for right ∈ {2,3}
        backward       = removing_index - 1  # -1 (truthy) for right ∈ {0,1}, 0 (falsy) for right ∈ {2,3}

        self.num_particles = 1
        self.positions     = np.delete(self.positions,     removing_index, axis=0)
        self.p             = np.delete(self.p,             removing_index, axis=0)
        self.current_force = np.delete(self.current_force, removing_index, axis=0)

        target_sign = 1 if right % 2 == 0 else -1  # basin: even→right(+), odd→left(-)


        while start_index != 0 :
            self.one_step()
            current_q = self.positions[0]
            start_index -= 1
            if backward:
                new_transit[new_len - start_index] = current_q
            else:
                new_transit[start_index] = current_q

            if self.point_in_stable_region(current_q):
                reached_wrong_basin = np.sign(current_q[0]) != target_sign
                new_transit = (new_transit[start_index : finish1_index + 1]
                            if right > 1 else
                            new_transit[finish1_index : new_len - start_index + 1])
                new_len = new_transit.shape[0]
                if reached_wrong_basin:

                    if return_Ns:
                        if right % 2 ==0:
                            N_a += 1
                        return original_transition_path, N_a, 2 - N_a

                    return original_transition_path
                if return_Ns:
                    if right % 2 !=0:
                        N_a += 1
                break

        self.num_particles = save_particle_num
        # ── Metropolis acceptance ─────────────────────────────────────────────

        if min(length_O / new_len, 1) > np.random.rand():
            if return_Ns:
                return new_transit, N_a, 2 - N_a
            return new_transit
        
        if return_Ns:
            return original_transition_path, N_a, 2 - N_a
        
        return original_transition_path

    def create_committor (self, measure_times=10, x_tiles=50, y_tiles=50):
        """Compute the empirical 2D committor probability grid using stochastic trial trajectories.

        This method discretizes the $[-3, 3] \times [-3, 3]$ spatial domain into a 2D meshgrid 
        of resolution `(x_tiles, y_tiles)`. From each grid coordinate, it initializes 
        `measure_times` independent particles with thermally sampled momenta drawn from a 
        Maxwell-Boltzmann distribution. 

        Trajectories are propagated step-by-step via `self.one_step()` until every particle 
        enters a stable region (as determined by `self.point_in_stable_region()`). The committor 
        probability $P_B(q)$ for each grid point $q = (x, y)$ is calculated as the fraction of 
        trial trajectories that commit to Basin B ($x > 0$).

        Parameters
        ----------
        measure_times : int, optional
            The number of stochastic trial trajectories launched per grid coordinate to 
            estimate commitment probabilities. Default is 10.
        x_tiles : int, optional
            Grid resolution along the x-axis ($\phi$), spanning the domain $[-3, 3]$. 
            Default is 50.
        y_tiles : int, optional
            Grid resolution along the y-axis ($\psi$), spanning the domain $[-3, 3]$. 
            Default is 50.

        Returns
        -------
        np.ndarray
            A 2D float array of shape `(x_tiles, y_tiles)` containing empirical committor 
            probabilities $P_B \in [0, 1]$.

        Notes
        -----
        * **Spatial Domain:** The grid boundaries are hardcoded to $[-3, 3]$ along both axes.
        * **Basin Definition:** A trajectory is counted as committing to **Basin B** if 
        it reaches a stable region with an x-coordinate strictly greater than zero 
        ($x > 0$). Otherwise, it is assumed to commit to Basin A ($x \le 0$).
        * **Dynamic Filtering:** Active trajectories are dynamically filtered as they 
        reach stable states (`~settled`), slicing `positions`, `p`, and `current_force` 
        to optimize iteration speed as particles settle.
        * **Attribute Mutation:** Temporarily overrides instance attributes (`self.num_particles`, 
        `self.positions`, `self.p`, and `self.current_force`) during calculation. 
        `self.num_particles` is restored to its original value upon completion.
        """
        # ── Build the grid ────────────────────────────────────────────────────
        x = np.linspace(-3, 3, x_tiles)
        y = np.linspace(-3, 3, y_tiles)
        X, Y     = np.meshgrid(x, y, indexing='ij')   # (x_tiles, y_tiles, 2)
        positions = np.stack((X, Y), axis=-1)          # shape (x_tiles, y_tiles, 2)

        committor  = np.empty((x_tiles, y_tiles))
        save_particle_num = self.num_particles
        for ix in range(x_tiles):
            for iy in range(y_tiles):
                start = positions[ix, iy]              # shape (2,)

                self.num_particles  = measure_times
                self.positions      = np.tile(start, (measure_times, 1))
                self.p              = self.m * np.random.normal(0, (self.m * self.temperature) ** 0.5,
                                        size=(measure_times, 2))
                self.current_force  = self.what_is_force(self.positions)

                reached_B = 0
                alive = np.ones(measure_times, dtype=bool)  # mask of unsettled particles

                while alive.any():
                    
                    self.one_step()

                    settled = self.point_in_stable_region(self.positions)
                    if not settled.any():
                        continue

                    reached_B              += np.sum(self.positions[settled, 0] > 0)
                    alive[alive]            = ~settled          # update global mask
                    self.num_particles      = int(alive.sum())
                    self.positions          = self.positions[~settled]
                    self.p                  = self.p[~settled]
                    self.current_force      = self.current_force[~settled]

                committor [ix, iy] = reached_B / measure_times
            print(f'Finished column {ix+1} of {x_tiles}')
        self.num_particles = save_particle_num
        return committor 

    def plot_committor (self, committor , plot_potential=True, colorbar = False):
        """Visualize a 2D committor probability grid with optional cell values and energy contours.

        This method takes a 2D array of committor probabilities $P_B$, maps its elements onto 
        a predefined 2D spatial domain ($[-3, 3]$ along the x-axis and $[-2.5, 2.5]$ along 
        the y-axis), and renders a 2D heatmap using `pcolormesh`.

        It supports two presentation modes for reading probabilities:
        1. **Colorbar Mode (`colorbar=True`):** Displays a continuous color scale bar alongside 
        the heatmap.
        2. **Text-Annotated Mode (`colorbar=False`):** Overlays numeric float labels formatted 
        to two decimal places (`f'{v:.2f}'`) in the center of every grid cell, using 
        adaptive text coloring for high contrast against the background.

        Optionally, it can evaluate `self.what_is_potential_energy()` over a fine grid to 
        overlay equipotential contour lines.

        Parameters
        ----------
        committor : np.ndarray
            A 2D float array of shape `(x_tiles, y_tiles)` containing committor probabilities 
            bounded between 0 and 1.
        plot_potential : bool, optional
            If True, computes the potential energy surface over a $[-4, 4] \times [-3, 3]$ 
            grid and overlays green equipotential contour lines. Default is True.
        colorbar : bool, optional
            If True, renders a colorbar legend. If False, annotates each individual grid cell 
            with its numerical value. Default is False.

        Returns
        -------
        None
            The method does not return any values; it directly renders and displays 
            the matplotlib figure.

        Notes
        -----
        * **Spatial Domain:** Bin centers are hardcoded to span $[-3, 3]$ for the x-axis 
        and $[-2.5, 2.5]$ for the y-axis.
        * **Edge Calculation:** Bin edges required by `plt.pcolormesh` are derived 
        automatically by extending bin centers outward by half a cell width ($\Delta x / 2$) 
        and height ($\Delta y / 2$).
        * **Adaptive Contrast Text:** When `colorbar=False`, text label color defaults to 
        `'white'` for cell values $v < 0.4$ and `'black'` for $v \ge 0.4$ to ensure 
        readability over the `'magma'` colormap.
        * **Aspect Ratio:** The plot enforces an equal aspect ratio (`ax.set_aspect('equal')`).
        """
        x_tiles, y_tiles = committor.shape
        x_centres = np.linspace(-3,   3,   x_tiles)
        y_centres = np.linspace(-2.5, 2.5, y_tiles)

        # pcolormesh needs edges, not centres — add half a cell on each side
        dx = x_centres[1] - x_centres[0]
        dy = y_centres[1] - y_centres[0]
        x_edges = np.append(x_centres - dx / 2, x_centres[-1] + dx / 2)
        y_edges = np.append(y_centres - dy / 2, y_centres[-1] + dy / 2)

        values = committor .T  # (y_tiles, x_tiles)

        fig, ax = plt.subplots(figsize=(8, 6))

        mesh = ax.pcolormesh(x_edges, y_edges, values, cmap='magma', vmin=0, vmax=1, shading='auto')
        if colorbar:
            fig.colorbar(mesh, ax=ax, label='Committor probability')

        else:
            for ix, xc in enumerate(x_centres):
                for iy, yc in enumerate(y_centres):
                    v = values[iy, ix]
                    text_color = 'white' if v < 0.4 else 'black'
                    ax.text(xc, yc, f'{v:.2f}', ha='center', va='center',
                            fontsize=5, color=text_color)

        if plot_potential:
            Xp, Yp = np.meshgrid(np.arange(-4, 4, 0.05), np.arange(-3, 3, 0.05), indexing='xy')
            points = np.stack((Xp, Yp), axis=-1)
            Z      = self.what_is_potential_energy(points)
            ax.contour(Xp, Yp, Z, levels=40, colors='green', linewidths=0.5, alpha=0.40)

        ax.set_xlim(x_edges[0], x_edges[-1])
        ax.set_ylim(y_edges[0], y_edges[-1])
        ax.set_xlabel('X coordinate')
        ax.set_ylabel('Y coordinate')
        ax.set_title('Committor probability')
        ax.set_aspect('equal')
        plt.tight_layout()
        plt.show()

# Machine Learning Part

## Preparing the training data.

In [ ]:

test = Transition_Path_Sampler(dimension = 2, molec_dynam_step = True, alpha = 50, temperature = 100, num_steps = 10**7, gamma = 10, m = 1, dt = 2e-3) 
print(test.simulate_extract_all_transition_paths(save=True, file_name= 'transition_paths_alpha=50_WQ', return_num_of_transitions=True))
test.plot_point_density()
test.load_trajectories('transition_paths_alpha=50_WQ run no 0')
transition_paths=[]
current_start_index = 0
transitions_counter = 0
last_index = test.trajectories.shape[0]
while current_start_index < last_index:
    transition_path, temp_start_index = test.extract_original_transition_path(test.trajectories[current_start_index: , :], return_last_index=True)
    if temp_start_index != None: #I could not think of a more efficient way than this little condition
        current_start_index += temp_start_index
        transition_paths.append(transition_path)
        transitions_counter += 1 
    else:
        current_start_index = last_index

print(f"Total transition paths extracted: {transitions_counter}")

with open("transition_paths_alpha=50_WQ.dat", "wb") as f:
    pickle.dump(transition_paths, f) 

flat = [item for sublist in transition_paths for item in sublist]
test.trajectories = np.array(flat)[:, np.newaxis, :]
test.plot_point_density()

for j in range(10**2):
    original_path_index = np.random.randint(0, len(transition_paths))
    original_path = transition_paths[original_path_index]
    for k in range(10**2):
        shoot = test.Create_a_transition_path_with_shooting(original_path)
        original_path = shoot.copy()
        transition_paths.append(original_path)

with open("transition_paths_alpha=50_WQ_V2.dat", "wb") as f:
    pickle.dump(transition_paths, f) 

flat = [item for sublist in transition_paths for item in sublist]
test.trajectories = np.array(flat)[:, np.newaxis, :]


test.plot_point_density()

    


### Committor creation

In [ ]:
test = Transition_Path_Sampler(molec_dynam_step = True, alpha = 100, temperature = 100, num_steps = 10**5, gamma = 500, m = 1, dt = 2e-3, num_particles = 20) 
test.set_initial_positions()
test.run()
test.plot_point_density()
committor  = test.create_committor (100, x_tiles=50, y_tiles=50) 
np.save('committor_for_alpha=100_T=100_gamma=500_m=1_WQ', committor) 
committor_grid = np.load('committor_for_alpha=100_T=100_gamma=500_m=1_WQ.npy') 
test.plot_committor (committor = committor_grid, colorbar=True)

## The network setup

In [ ]:
# ── 1. Neural Network ────────────────────────────────────────────────────────

class CommittorNet(nn.Module):
    """
    Maps a configuration q (dim-dimensional coordinate vector)
    to a scalar N(q).  The committor is then sigmoid(N(q)).

    Architecture: input → Linear(hidden_dims) + SiLU → Linear(1)
    SiLU (Swish) is chosen as the smooth activation function:
        SiLU(x) = x · sigmoid(x)
    It is everywhere differentiable, non-saturating for positive inputs,
    and empirically outperforms tanh/sigmoid in deep networks.
    """

    def __init__(self, input_dim: int, hidden_dims: list[int] = None):
        super().__init__()
        if hidden_dims is None:
            hidden_dims = [128] * 6
        
        layers = []
        in_dim = input_dim
        for n_hidden in hidden_dims:
            layers.append(nn.Linear(in_dim, n_hidden))
            layers.append(nn.SiLU())          # smooth activation without sigmoid problems (vanishing signal)
            in_dim = n_hidden

        layers.append(nn.Linear(in_dim, 1))   # scalar output N(q)

        self.net = nn.Sequential(*layers)

        # Weight initialisation: Xavier uniform keeps gradients well-scaled
        # through the deep stack
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, q: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ----------
        q : (batch, input_dim)

        Returns
        -------
        N_q : (batch,)   raw scalar output N(q)
        """
        return self.net(q).squeeze(-1)

    def save_weights(self, file_name: str = 'model_weights'):
        torch.save(self.state_dict(), file_name+'.pth')

    def load_weights(self, file_name: str = 'model_weights.pth'):
        self.load_state_dict(torch.load(file_name, weights_only=True))


# ── 2. Committor function ────────────────────────────────────────────────────

def committor(N_q: torch.Tensor) -> torch.Tensor:
    """
    P_B(q) = sigmoid( N(q) ) = 1 / (1 + exp(-N(q)))

    Guaranteed to lie in (0, 1).
    """
    return torch.sigmoid(N_q)


# ── 3. Selection probability ─────────────────────────────────────────────────

def selection_probability(
    N_q: torch.Tensor,
    lam: float = 1.0,
) -> torch.Tensor:
    """
    P_sel(q_s | X) = 1 / sum_{q_i in X} [ (N(q_s)^2 + λ^2) /
                                            (N(q_i)^2 + λ^2) ]

    Parameters
    ----------
    N_q_s   : (M,) –  N(q) for all configurations in the path X
    lam   : regularisation parameter λ

    Returns
    -------
    P_sel : same shape as N_q_s
    """
    lam2   = lam ** 2
    N_mod = N_q ** 2 + lam2
    normalisation = (1/N_mod).sum()
    return 1 / (normalisation * N_mod)


# ── 4. Expected number of transition paths ───────────────────────────────────

def n_tps_expected(P_B: torch.Tensor) -> torch.Tensor:
    """
    n_TPS_exp = Σ_i  2 · P_B(q_i) · (1 − P_B(q_i))

    Parameters
    ----------
    P_B : (k,)  committor values at k shooting points

    Returns
    -------
    scalar tensor
    """
    return 2.0 * (P_B * (1.0 - P_B)).sum()


# ── 5. Efficiency coefficient ────────────────────────────────────────────────

def efficiency_coefficient(
    n_tps_gen: float | torch.Tensor,
    n_tps_exp: torch.Tensor,
) -> torch.Tensor:
    """
    α_eff = min( 1,  (1 − n_TPS_gen / n_TPS_exp)^2 )

    Parameters
    ----------
    n_tps_gen : number of transition paths actually generated
    n_tps_exp : expected number (output of n_tps_expected)

    Returns
    -------
    scalar tensor in [0, 1]
    """
    ratio = n_tps_gen / n_tps_exp
    alpha = (1.0 - ratio) ** 2
    return torch.clamp(alpha, max=1.0)


# ── 6. Binomial negative log-likelihood loss ─────────────────────────────────

def binomial_nll_loss(
    P_B: torch.Tensor,
    n_A: torch.Tensor,
    n_B: torch.Tensor,
    eps: float = 1e-8,
) -> torch.Tensor:
    """
    Negative log-likelihood of the binomial shooting outcome.

    −ln ∏_i p(n_A_i, n_B_i | q_i)

    where  p(n_A, n_B | q) = C(n_A+n_B, n_B)
                              · (1−P_B)^n_A · P_B^n_B

    The log-binomial-coefficient is computed via the log-gamma function
    (stable for non-integer counts too):
        ln C(n,k) = ln Γ(n+1) − ln Γ(k+1) − ln Γ(n−k+1)

    Parameters
    ----------
    P_B : (k,)   committor at each shooting point
    n_A : (k,)   number of A-terminating trajectories  (float or int tensor)
    n_B : (k,)   number of B-terminating trajectories

    Returns
    -------
    scalar loss
    """
    n_A  = n_A.float()
    n_B  = n_B.float()
    n    = n_A + n_B

    # log C(n, n_B)  –  numerically stable via log-gamma
    log_binom = gammaln(n + 1) - gammaln(n_B + 1) - gammaln(n_A + 1)

    # log-likelihood per shooting point
    log_lik = (
        log_binom
        + n_A * torch.log(1.0 - P_B + eps)
        + n_B * torch.log(P_B + eps)
    )

    return -log_lik.sum()


# ── 7. Training loop ─────────────────────────────────────────────────────────

def train_step(
    model:     CommittorNet,
    q_configs: torch.Tensor,      # (K, input_dim)  – all K shooting points
    n_A:       torch.Tensor,      # (K,)
    n_B:       torch.Tensor,      # (K,)
    n_tps_gen: int,               # actual reactive paths counted this iteration
    optimizer: optim.Optimizer,
    scheduler: optim.lr_scheduler._LRScheduler = None,
    alpha_eff_threshold: float = 0.3,
    device:    torch.device = torch.device("cpu"),
) -> dict:
    """
    One learning iteration.

    Steps
    -----
    1.  Forward pass → N(q), P_B(q)
    2.  Compute α_eff; skip backprop if α_eff < threshold
    3.  Backprop on binomial NLL loss

    Returns
    -------
    dict with keys: loss, alpha_eff, n_tps_exp, P_B
    """
    model.train()                               # set model to training mode
    q_configs = q_configs.to(device)
    n_A       = n_A.to(device)
    n_B       = n_B.to(device)

    N_q = model(q_configs)                      # (K,)
    P_B = committor(N_q)                        # (K,)

    n_exp = n_tps_expected(P_B)
    alpha = efficiency_coefficient(n_tps_gen, n_exp)

    result = {
        "alpha_eff": alpha.item(),
        "n_tps_exp": n_exp.item(),
        "P_B":       P_B.detach().cpu()
    }

    if alpha.item() < alpha_eff_threshold:
        print(f"  α_eff = {alpha.item():.4f} < {alpha_eff_threshold} → skipping update")
        return result

    loss = binomial_nll_loss(P_B, n_A, n_B)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if not (scheduler is  None):
        scheduler.step(loss)

    return result

# ── 8. Evaluation function ──────────────────────────────────────────────────

def evaluate_model(model: CommittorNet, committor_grid: np.ndarray = None, print_min_max: bool = False, return_min_max: bool = False):
    """Evaluate a neural network committor model over a 2D spatial grid and visualize predictions or residuals.
    
    This function passes a 2D meshgrid of coordinates through a trained `CommittorNet` 
    model to compute raw model outputs $N(q)$, applies a sigmoid transformation to map 
    outputs to committor probabilities $P_B = \frac{1}{1 + e^{-N(q)}}$, and renders a 2D 
    heatmap of the results. 

    Depending on whether a reference `committor_grid` is supplied, the function operates 
    in one of two modes:
    1. **Standalone Prediction Mode (`committor_grid=None`):** Evaluates the model on an 
        $80 \times 80$ grid over the domain $[-\pi, 0] \times [-\pi, 0]$ (typically 
        representing $\phi / \psi$ dihedral space) and plots $P_B$.
    2. **Reference Comparison Mode (`committor_grid` provided):** Evaluates the model on 
        a grid over $[-3, 3] \times [-3, 3]$ matching the dimensions of `committor_grid`, 
        computes the residual difference ($P_{B, \text{model}} - P_{B, \text{ref}}$), and 
        plots a difference heatmap.

    Parameters
    ----------
    model : CommittorNet
        A PyTorch neural network predicting unnormalized committor values $N(q)$ 
        for 2D input coordinates $q = (\phi, \psi)$.
    committor_grid : np.ndarray or None, optional
        A 2D array of reference committor probabilities $P_B \in [0, 1]$. If provided, 
        the function computes and visualizes the difference between model predictions 
        and this reference grid. Default is `None`.
    print_min_max : bool, optional
        If `True` and `committor_grid` is provided, prints the minimum and maximum 
        residual differences between predicted and reference $P_B$ values. Default is `False`.
    return_min_max : bool, optional
        If `True` and `committor_grid` is provided, returns the minimum and maximum 
        residual differences as a tuple. Default is `False`.

    Returns
    -------
    tuple of (float, float) or None
        - If `return_min_max=True` and `committor_grid` is provided: Returns 
            `(min_diff, max_diff)`, ignoring `NaN` values.
        - Otherwise: Returns `None`.

    Raises
    ------
    ValueError
        If `committor_grid` is supplied but its shape does not match the 2D meshgrid 
        generated from its dimensions.

    Notes
    -----
    * **Device Handling:** Inputs are automatically moved to the device of the model's 
        parameters (CPU or GPU).
    * **Zero Difference Handling:** In comparison mode, exact zero differences 
        ($P_{B, \text{model}} - P_{B, \text{ref}} = 0$) are masked as `np.nan` prior to 
        visualization and min/max calculation to highlight true variance.
    * **Colormaps:** Standalone predictions are plotted using the `'magma'` colormap 
        over $[0, 1]$, whereas residual differences are plotted using the divergent `'brg'` 
        colormap over $[-1, 1]$.
    """
    model.eval()
    with torch.no_grad():
        if committor_grid is None:
            x = torch.linspace(-3, 3, 50)
            y = torch.linspace(-3, 3, 50)
        else:
            x = torch.linspace(-3, 3, committor_grid.shape[0])
            y = torch.linspace(-3, 3, committor_grid.shape[1])
        X, Y = torch.meshgrid(x, y, indexing='ij') # create a grid of points in the configuration space
        # pass the grid through the model to get N(q) values, then reshape back to grid form
        q = torch.stack((X.flatten(), Y.flatten()), dim=-1).to(next(model.parameters()).device) 
        N_q = model(q).cpu().numpy().reshape(X.shape)
        P_b = 1/(1 + np.exp(-N_q))
        if committor_grid is None:
            values = P_b .T  # (y_tiles, x_tiles)

            plt.figure(figsize=(8, 6))

            im = plt.pcolormesh(x, y, values, cmap='magma', vmin=0, vmax=1, shading='auto')
            plt.colorbar(im, label='Committor probability')
            plt.xlabel('X coordinate')
            plt.ylabel('Y coordinate')
            plt.title('Committor probability')
            plt.tight_layout()
            plt.show()
            if return_min_max or print_min_max:
                print("There is nothing to compare the model with.")
        else:
            if committor_grid.shape != P_b.shape:
                raise ValueError("Provided committor_grid shape does not match model output shape.")
            diff = P_b - committor_grid
            diff = np.where(diff == 0, np.nan, diff)  # Set zero differences to NaN for better visualization
            if print_min_max:
                print(f"The maximal difference between the model and the reference committor is {np.nanmax(diff)}")
                print(f"The minimal difference between the model and the reference committor is {np.nanmin(diff)}")
            if return_min_max:
                return np.nanmin(diff), np.nanmax(diff)
            plt.figure(figsize=(8, 6))
            im = plt.pcolormesh(x, y, diff.T, cmap='brg', vmin=-1, vmax=1, shading='auto')
            plt.colorbar(im, label='Model P_B - Reference P_B')
            plt.xlabel('X coordinate')
            plt.ylabel('Y coordinate')
            plt.title('Difference between Model and Reference Committor')
            plt.tight_layout()
            plt.show()
            
            
def plot_isocommittor_line(committor_grid: np.ndarray, num_lines: int = 10, alpha_of_simul_committor = 50, plot_stable_states = True):
    """Visualize isocommittor lines and stable states on a 2D probability grid.

    This function generates a contour plot of a provided 2D committor probability grid 
    over the predefined spatial domain $[-\pi, 0] \times [-\pi, 0]$ (representing 
    $\phi$ and $\psi$ dihedral angles). It draws evenly spaced isocommittor lines 
    (contours of constant probability) and specifically highlights the boundary states: 
    State A ($P_B \approx 0$) in blue, and State B ($P_B \approx 1$) in red.

    Additionally, it can optionally overlay a scatter plot of the exact stable regions 
    by evaluating an external `angles_in_stable_region` mask over the grid.

    Parameters
    ----------
    committor_grid : np.ndarray
        A 2D array of committor probabilities, typically bounded between 0 and 1. The 
        shape of this array dictates the resolution of the generated spatial meshgrid.
    num_lines : int, optional
        The number of evenly spaced contour levels to draw between 0 and 1 (inclusive). 
        Default is 10.
    alpha_of_simul_committor : int or float, optional
        Currently unused in the function body. Maintained for signature compatibility. 
        Default is 50.
    plot_stable_states : bool, optional
        If True, evaluates the external `angles_in_stable_region` function over the 
        meshgrid and overlays cyan scatter points to explicitly denote the stable 
        basins. Default is True.

    Returns
    -------
    None
        The function does not return any values; it directly renders and displays 
        the matplotlib figure.

    Notes
    -----
    * **Spatial Domain:** The grid coordinates are strictly hardcoded to span from 
      $-\pi$ to $0$ for both the x-axis ($\phi$) and y-axis ($\psi$). 
    * **Dependencies:** This function relies on `torch.linspace` to generate the 
      coordinate vectors before passing them to `np.meshgrid`. It also requires the 
      external `angles_in_stable_region` callable to be defined in the global scope 
      if `plot_stable_states=True`.
    * **Contour Highlighting:** Boundary states are detected using `np.isclose` with 
      default tolerances against 0 and 1.
    """
    x = np.linspace(-3, 3, committor_grid.shape[0])
    y = np.linspace(-3, 3, committor_grid.shape[1])
    X, Y = np.meshgrid(x, y, indexing='ij')
    plt.figure(figsize=(8, 6))
    points = np.stack((X, Y), axis=-1)  # shape (len(y), len(x), 2)
    
    problem = Transition_Path_Sampler(dimension = 2, molec_dynam_step = True, alpha = alpha_of_simul_committor, temperature = 100, num_steps = 10**7, gamma = 25, m = 10, dt = 2e-3)
    Z = problem.what_is_potential_energy(points)
    plt.contour(X, Y, Z, levels=40, colors='gray', linestyles='dashed',linewidths=0.5, alpha=0.3)

    levels = np.linspace(0, 1, num_lines) 
    contour = plt.contour(X, Y, committor_grid, levels=levels, colors='green')
    plt.clabel(contour, inline=True, fontsize=8)
    plt.contour(X, Y, np.isclose(committor_grid, 1), colors='red', linewidths=2)  # Highlight the stable state B
    plt.contour(X, Y, np.isclose(committor_grid, 0), colors='blue', linewidths=2)  # Highlight the stable state A

    if plot_stable_states:
        stable_mask = problem.point_in_stable_region(points)
        plt.scatter(X[stable_mask], Y[stable_mask], color='cyan', label='Stable Region ', s=5)
    plt.xlabel('X coordinate')
    plt.ylabel('Y coordinate')
    plt.title('Isocommittor Lines')
    plt.tight_layout()
    plt.show()

def plot_confusion_plot(committor_grid: np.ndarray, model: torch.nn.Module, threshold: float = 0.05, return_matches: bool = False):
    """Generate a sorted fidelity/confusion plot comparing model-predicted committor probabilities against reference values.

    This function evaluates a PyTorch neural network model across a 2D spatial grid spanning 
    $[-3, 3] \times [-3, 3]$ matching the dimensions of `committor_grid`. The raw model 
    outputs $N(q)$ are transformed into predicted committor probabilities $P_B$ via an 
    external `committor()` function.

    To visualize model fidelity:
    1. Grid points are flattened and sorted in ascending order of the reference $P_B$ values.
    2. Model predictions are classified based on whether their absolute error 
       $|P_{B, \text{pred}} - P_{B, \text{ref}}|$ falls within the specified `threshold`.
    3. Points within tolerance are highlighted in green, while deviating points are rendered 
       in red alongside a shaded $\pm\text{threshold}$ envelope.

    Parameters
    ----------
    committor_grid : np.ndarray
        A 2D array containing the reference ground-truth committor probabilities $P_B \in [0, 1]$. 
        Its shape dictates the resolution of the evaluated 2D spatial grid.
    model : torch.nn.Module
        A PyTorch neural network that takes 2D coordinate inputs $(\phi, \psi)$ and 
        outputs raw unnormalized committor network values $N(q)$.
    threshold : float, optional
        The maximum allowable absolute difference between predicted $P_B$ and reference $P_B$ 
        for a grid point to be considered a match. Default is 0.05.
    return_matches : bool, optional
        If `True`, skips plot rendering and returns the total count of grid points 
        whose predictions fall within the specified `threshold`. Default is `False`.

    Returns
    -------
    int or None
        - If `return_matches=True`: Returns the integer count of matching grid points.
        - If `return_matches=False`: Returns `None` and displays the matplotlib figure.

    Notes
    -----
    * **Spatial Domain:** The spatial meshgrid coordinates are hardcoded to span from 
      $-3$ to $3$ in both axes ($x$ and $y$).
    * **Dependencies:** This function expects an external `committor(N)` function or callable 
      to be available in the global scope to map the model's output tensor to $[0, 1]$ 
      probabilities.
    * **Device Compatibility:** Input coordinates are automatically moved to the device 
      hosting the model's parameters (CPU or CUDA).
    * **Visualization Mechanics:** Sorting the points by increasing reference $P_B$ maps 
      the 2D spatial domain into a 1D curve, making systematic model over- or 
      under-predictions easy to identify across the entire transition region.
    """
    x = np.linspace(-3, 3, committor_grid.shape[0])
    y = np.linspace(-3, 3, committor_grid.shape[1])
    X, Y = np.meshgrid(x, y, indexing='ij')
    
    points = np.stack((X, Y), axis=-1)  # shape (len(y), len(x), 2)
    device = next(model.parameters()).device 
    with torch.no_grad():
        predicted_N_device = model(torch.tensor(points, dtype=torch.float32).to(device))
        predicted_P_B_device = committor(predicted_N_device)
        predicted_P_B = predicted_P_B_device.cpu().numpy().flatten() #shape (len(x)*len(y),)
    true_B = committor_grid.flatten() #shape (len(x)*len(y),)


    x = np.arange(len(predicted_P_B))  # Index for each point in the grid
    
    sorted_indices = np.argsort(true_B)
    true_B = true_B[sorted_indices]
    predicted_P_B = predicted_P_B[sorted_indices]

    TP_mask = (np.abs(predicted_P_B - true_B) <= threshold)
    
    if return_matches:
        return np.sum(TP_mask)
    
    FP_mask = ~TP_mask
    
    plt.figure(figsize=(12, 9))
    plt.fill_between(x, true_B - threshold, true_B + threshold, color='green', alpha=0.2, label='±1 threshold')
    plt.scatter(x[TP_mask], predicted_P_B[TP_mask], color='green', label='Fits to reference', s=5)
    plt.scatter(x[FP_mask], predicted_P_B[FP_mask], color='red', label='Deviation from reference', s=5)

    plt.xlabel('Order of mesh grid point')
    plt.ylabel('Committor Probability')
    plt.title(f'Confusion Plot at allowed error {threshold}')
    plt.legend()
    plt.tight_layout()
    plt.show()

## Training part 

with already set values that can be set arbitrarily.

In [ ]:
# ── device selection ────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {device}\n")

# ── hyper-parameters ────────────────────────────────────────────────────
INPUT_DIM   = 2      # e.g. (x, y, z) Cartesian coordinates
HIDDEN_DIM  =  [160, 176, 144, 32, 32]  # number of neurons in each hidden layer [192, 128, 32, 128, 32, 32] [256, 128, 64, 128, 64, 32, 16, 8, 4] [256, 64, 32, 64, 32, 128], [128, 32, 16, 12, 16, 32], [192, 128, 32, 128, 32, 32]    
K           = 272     # shooting points per iteration 
LAM         = 5
LR          = 5e-3
N_ITER      = 200
ALPHA_THRESH = (1 - (K - np.log10(K) + 2) / K)**2 # only update if α_eff > 1 - (K-2)/K, i.e. if the fail is max 2 transition paths per iteration 


# ── model + optimiser ───────────────────────────────────────────────────
print(model, "\n")
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}\n")
TPS = Transition_Path_Sampler(dimension = INPUT_DIM, molec_dynam_step = True, alpha = 50, temperature = 100, num_steps = 10**7, gamma = 10, m = 1, dt = 2e-3)     
with open("transition_paths_alpha=50_WQ_V2.dat", "rb") as f:
    transition_paths = pickle.load(f)

committor_grid = np.load('committor_for_alpha=50_T=100_gamma=500_m=10_WQ.npy')

model = CommittorNet(INPUT_DIM, HIDDEN_DIM).to(device) 
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',      # we minimize loss
    factor=0.5,      # LR ← LR / 2
    patience=30,     # wait 30 epochs with no improvement
)

sampled_indices = torch.randint(0, len(transition_paths), (N_ITER,)) #randomly sample K transition paths from the reference file
los = np.empty(N_ITER)

for it_idx, it in enumerate(sampled_indices):
    path_idx = it.item()
    path     = transition_paths[path_idx]                                   # numpy array (M, input_dim)
    path_tensor = torch.tensor(path, dtype=torch.float32).to(device)       # (M, input_dim)

    # ── Select K shooting points (no gradients needed here) ──────────────
    with torch.no_grad():
        N_s       = model(path_tensor)                                      # (M,)
        sel_probs = selection_probability(N_s.cpu(), lam=LAM)              # (M,) on CPU

    selected_index = torch.multinomial(sel_probs, num_samples=1).item()  # (1,)

    # ── Run K two-way shooting attempts, collect results ─────────────────
    q_list  = []
    n_A_list = []
    n_B_list = []
    n_tps_gen = 0                                       # fix: always initialised

    for trial in range(K):
        new_path, n_A_i, n_B_i = TPS.Create_a_transition_path_with_shooting(
            path, shooting_point_idx=selected_index, return_Ns=True)
        q_list.append(path[selected_index])
        n_A_list.append(n_A_i)
        n_B_list.append(n_B_i)
        # Reactive = one half-trajectory reaches A AND the other reaches B              
        if not np.array_equal(new_path, path):  #if it is new path we shoot from this and replace the old path.

            n_tps_gen += 1 #it could be made as condition
            path = new_path.copy()  # update path for next attempt 
            path_tensor = torch.tensor(path, dtype=torch.float32).to(device)  # update tensor for next attempt
            with torch.no_grad():
                N_s       = model(path_tensor)                                      # (M,)
                sel_probs = selection_probability(N_s.cpu(), lam=LAM)              # (M,) on CPU
        #in case that path was not updated there is no need to recompute the selection probabilities, because we did'n performed learning step yet
        selected_index = torch.multinomial(sel_probs, num_samples=1).item() # try shooting from a different point on the same or updated path

    # ── Build batched tensors (K, ...) and run one training step ─────────
    q_configs = torch.tensor(np.stack(q_list), dtype=torch.float32)    # (K, input_dim)
    n_A_tensor = torch.tensor(n_A_list, dtype=torch.int)           # (K,)
    n_B_tensor = torch.tensor(n_B_list, dtype=torch.int)           # (K,)

    res = train_step(
        model, q_configs, n_A_tensor, n_B_tensor,
        n_tps_gen=n_tps_gen,
        optimizer=optimizer,
        alpha_eff_threshold=ALPHA_THRESH,
        device=device,
        scheduler=scheduler,
    )
    los[it_idx] = res['alpha_eff']

#plotting the learning curve
plt.figure(figsize=(8, 6))
plt.plot(range(N_ITER), los, 'x-')
plt.axhline(ALPHA_THRESH, color='red', linestyle='--')
plt.title(f'Learning curve for {N_ITER} iterations')
plt.xlabel('Iteration')
plt.ylabel('Effective Alpha')
plt.tight_layout()
plt.show()

evaluate_model(model)
evaluate_model(model, committor_grid=committor_grid, print_min_max=True)

## Save the weights

In [ ]:
model.save_weights('committor_model_final_alpha=50_[160, 176, 144, 32, 32]_hidden_dims_K=272')

An example of the usage of methods plot_isocommittor_line and plot_confusion_plot

In [ ]:
committor_grid_model = np.empty_like(committor_grid)
x = torch.linspace(-3, 3, committor_grid.shape[0])
y = torch.linspace(-3, 3, committor_grid.shape[1])
X, Y = torch.meshgrid(x, y, indexing='ij')
q = torch.stack((X.flatten(), Y.flatten()), dim=-1).to(device)
with torch.no_grad():
    N_q = model(q).cpu().numpy().reshape(X.shape)
    P_b = 1 / (1 + np.exp(-N_q))
plot_isocommittor_line(committor_grid=P_b, num_lines=10, alpha_of_simul_committor=50)


plot_confusion_plot(committor_grid=committor_grid, model=model, threshold=0.05)

# Optuna optimisation

In [ ]:
import optuna
from optuna.trial import Trial
from torch.utils.data import DataLoader

def objective(
    reference_TPS: Transition_Path_Sampler,
    trial: Trial,
    train_loader: DataLoader, #Removed val_loader because calculating this was too expensive.
    val_loader: DataLoader,
    input_dim: int,
    device: torch.device,
    #ALPHA_THRESH: float,
    #batch_size: int,

) -> float:
    """
    Optuna objective function.

    Returns:
        Validation MAE on normalized targets (lower is better).
    """
    # Hyperparameters to tune
    learning_rate = 5e-3
    hidden_layers = trial.suggest_int("hidden_layers", 5, 7)
    batch_size = trial.suggest_int("batch_size", 16, 256+16, step=32)
    ALPHA_THRESH = (1 - (batch_size - np.log10(batch_size) + 2) / batch_size)**2 #for 100 --> 4, 10 -->3,  mistakes per iteration are allowed
    architecture = {}
    for i in range(hidden_layers):
        architecture[f"hidden_width_{i}"] = trial.suggest_int(f"hidden_width_{i}", 16, 192, step=16)
    
    num_epochs = 150
    LAM = trial.suggest_categorical("LAM", [0.01, 0.1, 1.0, 5.0, 10.0])

    HIDDEN_DIM = list(architecture.values())
    model = CommittorNet(input_dim, HIDDEN_DIM).to(device) 
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    best_val = 0

    sampled_indices = torch.randint(0, len(train_loader), (num_epochs,)) #randomly sample K transition paths from the reference file
    for epoch, it in enumerate(sampled_indices):
        path_idx = it.item()
        path     = train_loader[path_idx]                                   # numpy array (M, input_dim)
        path_tensor = torch.tensor(path, dtype=torch.float32).to(device)       # (M, input_dim)

        # ── Select batch_size shooting points (no gradients needed here) ──────────────
        with torch.no_grad():
            N_s       = model(path_tensor)                                      # (M,)
            sel_probs = selection_probability(N_s.cpu(), lam=LAM)              # (M,) on CPU

        selected_index = torch.multinomial(sel_probs, num_samples=1).item()  # (1,)

        # ── Run batch_size two-way shooting attempts, collect results ─────────────────
        q_list  = []
        n_A_list = []
        n_B_list = []
        n_tps_gen = 0                                       # fix: always initialised

        for shoot_trial in range(batch_size):
            new_path, n_A_i, n_B_i = reference_TPS.Create_a_transition_path_with_shooting(
                path, shooting_point_idx=selected_index, return_Ns=True)
            q_list.append(path[selected_index])
            n_A_list.append(n_A_i)
            n_B_list.append(n_B_i)

            # Reactive = one half-trajectory reaches A AND the other reaches B              

            if not np.array_equal(new_path, path):  #if it is new path we shoot from this and replace the old path.
                n_tps_gen += 1 #it could be made as condition
                path = new_path.copy()  # update path for next attempt 
                path_tensor = torch.tensor(path, dtype=torch.float32).to(device)  # update tensor for next attempt
                with torch.no_grad():
                    N_s       = model(path_tensor)                                      # (M,)
                    sel_probs = selection_probability(N_s.cpu(), lam=LAM)              # (M,) on CPU
            #in case that path was not updated there is no need to recompute the selection probabilities, because we did'n performed learning step yet
            selected_index = torch.multinomial(sel_probs, num_samples=1).item() # try shooting from a different point on the same or updated path

        # ── Build batched tensors (batch_size, ...) and run one training step ─────────
        q_configs = torch.tensor(np.stack(q_list), dtype=torch.float32)    # (batch_size, input_dim)
        n_A_tensor = torch.tensor(n_A_list, dtype=torch.int)           # (batch_size,)
        n_B_tensor = torch.tensor(n_B_list, dtype=torch.int)           # (batch_size,)

        res = train_step(
            model, q_configs, n_A_tensor, n_B_tensor,
            n_tps_gen=n_tps_gen,
            optimizer=optimizer,
            alpha_eff_threshold=ALPHA_THRESH,
            device=device,
        )
        val_metric = plot_confusion_plot(committor_grid=val_loader, model=model, threshold=0.05, return_matches=True)
        best_val = max(best_val, val_metric)

        # Report intermediate metric for pruning
        trial.report(val_metric, step=epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()
    
    plot_confusion_plot(committor_grid=val_loader, model=model, threshold=0.05)
    return best_val


# ── Optuna study setup ────────────────────────────────────────────────────
pruner = optuna.pruners.MedianPruner(
        n_startup_trials=5,
        n_warmup_steps=5,
        interval_steps=1,
    )
study = optuna.create_study(
        direction="maximize",  # we want to maximize the number of matches in the confusion plot
        pruner=pruner,
        study_name="pytorch_mlp_optimization",
    )
INPUT_DIM   = 2
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {device}\n")
TPS = Transition_Path_Sampler(dimension = INPUT_DIM, molec_dynam_step = True, alpha = 100, temperature = 100, num_steps = 10**7, gamma = 10, m = 1, dt = 2e-3) 
with open("transition_paths_alpha=100_WQ_V2.dat", "rb") as f:
    transition_paths = pickle.load(f) #train_loader
committor_grid = np.load('committor_for_alpha=100_T=100_gamma=500_m=1_WQ.npy') #val_loader
study.optimize(
        lambda trial: objective(
            reference_TPS = TPS,
            trial=trial,
            train_loader=transition_paths,
            val_loader=committor_grid,
            input_dim=INPUT_DIM,
            device=device,
        ),
        n_trials=150,
        timeout=7200,
        show_progress_bar=True,
    )

print("\nOptimization complete.")
print(f"Best trial number: {study.best_trial.number}")
print(f"Best validation MAE: {study.best_value:.6f}")
print("Best hyperparameters:")
for key, value in study.best_trial.params.items():
    print(f"  {key}: {value}")